# Hypothesis Testing: Automatidata project 

**Note:** For the purpose of this exercise, assume that the sample data comes from an experiment in which customers are randomly selected and divided into two groups: 1) customers who are required to pay with credit card, 2) customers who are required to pay with cash. Without this assumption, we cannot draw causal conclusions about how payment method affects fare amount.

## **Conduct an A/B test**


In [18]:
# Import packages & libraries to compute descriptive statistics 
# and conduct a hypothesis test.
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [3]:
# Load dataset into dataframe
rides_df = pd.read_csv("Clean_2017_Taxi_Trip_Data.csv", index_col = 0)

### Task 2. Data exploration

In [4]:
rides_df.describe()

,VendorID,trip_duration,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,tip_amount,tolls_amount,total_amount
count,22698.000000,22698.000000,22698.000000,22698.000000,22698.000000,22698.000000,22698.000000,22698.000000,22698.000000,22698.000000,22698.000000,22698.000000
mean,1.556260,17.014605,1.643801,2.913441,1.039078,162.407877,161.523482,1.336902,13.031005,1.835862,0.312555,16.307784
std,0.496836,61.997769,1.283959,3.653200,0.281169,66.631429,70.137938,0.496217,13.205886,2.800661,1.399241,16.092441
min,1.000000,-17.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,-120.300000
25%,1.000000,6.600000,1.000000,0.990000,1.000000,114.000000,112.000000,1.000000,6.500000,0.000000,0.000000,8.750000
50%,2.000000,11.200000,1.000000,1.610000,1.000000,162.000000,162.000000,1.000000,9.500000,1.350000,0.000000,11.800000
75%,2.000000,18.400000,2.000000,3.060000,1.000000,233.000000,233.000000,2.000000,14.500000,2.450000,0.000000,17.800000
max,2.000000,1439.600000,6.000000,33.960000,5.000000,265.000000,265.000000,4.000000,999.990000,200.000000,19.100000,1200.290000


In [41]:
conclusive_res = rides_df.loc[(rides_df.total_amount > 0), 'total_amount']
conclusive_res.quantile(0.997) # majority of N.D. range

82.80963

In [5]:
(rides_df[['payment_type', 'fare_amount']].groupby(by='payment_type', as_index=False)
 .agg(fare_amount = pd.NamedAgg(column='fare_amount', aggfunc='mean'),
      count = pd.NamedAgg(column='payment_type', aggfunc='count'))
)

,payment_type,fare_amount,count
0,1,13.425570,15264
1,2,12.213546,7267
2,3,12.367934,121
3,4,12.989130,46


Based on the averages shown, it appears that customers who pay in credit card tend to pay a larger fare amount than customers who pay in cash. However, this difference might arise from random sampling, rather than being a true difference in fare amount.

### Task 3a. Problem Definition & Hypothesis

* **Goal:** Discover if customers who use credit cards pay higher fare amounts than customers who use cash.
* **Hypotheses:**
    * $H_0$: There is no difference in the average fare amount between customers who use credit cards and customers who use cash.

    * $H_A$: There is a difference in the average fare amount between customers who use credit cards and customers who use cash.

Determine if the difference in mean of fare amount in relation to the payment type is truly significant.  

In [16]:
# Conduct a two-sample t-test.
credit_sample = rides_df[rides_df.payment_type == 1]
cash_sample = rides_df[rides_df.payment_type == 2]

res = sm.stats.ttest_ind(a=credit_sample['fare_amount'], b=cash_sample['fare_amount'], equal_var=False)
print(f'p-value for two-sided test: {res.pvalue:.4g}')

p-value for two-sided test: 6.797e-12


**Since the p-value < 0.05, there is sufficient evidence to conclude that the difference in mean of fare amount btw customers who pay by credit card & cash is real.** 

### Task 3b. Experiment Design

Determine the minimum required sample size for detecting a significant difference if a new payment method (eg. QR, e-credit, etc.) is introduced. 
* Assumptions:
    - Current Mean Revenue = \\$15.98  
    - Expected Revenue with new payment method = 17.58 (10% increase)
    - Confidence interval = 95%
    - Statistical Power = 80%
 
which aims to improve the drivers' income by 10% (Monthly: \\$30,861.1 * 1.1). 

In [45]:
# Conduct a Power Test
mean_rev = conclusive_res.loc[conclusive_res < 82.8].mean()
current_rev, expected_rev = mean_rev, mean_rev * 1.1 
SD = conclusive_res.std()

effect_size = (expected_rev - current_rev) / SD
analysis = sm.stats.TTestIndPower()
sample_size = analysis.solve_power(effect_size=effect_size, alpha=0.05,
                                   power=0.8, alternative='larger')
print(f'n= {int(sample_size)}')

n= 1250


In [47]:
SD

16.063991091512626

**Result:** Each group requires at least 1250 users. 

### Task 4. Communicate insights with stakeholders

1. Assuming the general population of customers possess both payment types but tend to pay by cash or credit card, we can conclude that credit card is always preferred over cash by customers at high fare amounts. This may be due to the security & convenience of holding a credit card than cash.    

2. First, an A/B test should only be performed in a randomized controlled experiment, inhibiting the risks of other nuisance factors such as age, gender, social status, etc.
Second, this project makes the assumption of customers possessing both payment type but tend to pay by cash or credit card, which is often not true.    

With the profound transition into digital era, e-credit wallet should be introduced as one of the additional payment type to help improve the revenue for taxi cab drivers.   